# Data Preparation
We downloaded daily OHLCV data for 14 equities across 6 sectors from yfinance (2019–2024). 
This notebook validates the raw data, checks for issues, and standardizes it for modeling.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from scipy import stats
from statsmodels.tsa.stattools import adfuller

In [2]:
df_raw = pd.read_csv("../data/raw/AAPL.csv")

In [3]:
print (df_raw.head)

<bound method NDFrame.head of             Date       Close        High         Low        Open     Volume
0     2019-01-02   37.469196   37.689856   36.593681   36.750278  148158800
1     2019-01-03   33.737007   34.574560   33.691926   34.161714  365248800
2     2019-01-04   35.177208   35.246017   34.118999   34.292203  234428400
3     2019-01-07   35.098911   35.312454   34.617259   35.281608  219111200
4     2019-01-08   35.768005   36.021883   35.238901   35.485657  164101200
...          ...         ...         ...         ...         ...        ...
1504  2024-12-23  253.649429  254.027007  251.840976  253.152604   40858800
1505  2024-12-24  256.560822  256.570737  253.669277  253.868019   23234700
1506  2024-12-26  257.375610  258.448771  255.994450  256.550893   27237100
1507  2024-12-27  253.967392  257.057664  251.453455  256.193162   42355300
1508  2024-12-30  250.598907  251.890657  249.158116  250.628716   35557500

[1509 rows x 6 columns]>


In [4]:
print(df_raw.shape)

(1509, 6)


In [5]:
print(df_raw.columns.tolist())
print(df_raw.isnull().sum())

['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [6]:
for ticker in os.listdir("../data/raw/"):
    if ticker.endswith(".csv"):
        df = pd.read_csv(f"../data/raw/{ticker}")
        name = ticker.replace(".csv", "")
        print(f"{name}: {df.shape} | nulls: {df.isnull().sum().sum()}")

AAPL: (1509, 6) | nulls: 0
AMD: (1509, 6) | nulls: 0
AMZN: (1509, 6) | nulls: 0
BAC: (1509, 6) | nulls: 0
CVX: (1509, 6) | nulls: 0
GOOGL: (1509, 6) | nulls: 0
GS: (1509, 6) | nulls: 0
JNJ: (1509, 6) | nulls: 0
JPM: (1509, 6) | nulls: 0
MSFT: (1509, 6) | nulls: 0
NVDA: (1509, 6) | nulls: 0
PFE: (1509, 6) | nulls: 0
WMT: (1509, 6) | nulls: 0
XOM: (1509, 6) | nulls: 0


All 14 tickers have exactly 1509 rows and zero null values. yfinance with auto_adjust=True. Handles splits and dividend adjustments automatically, which explains the clean output.

CHECKING FOR OTHER INCONSISTENCIES IN DATA

In [7]:
for file in os.listdir("../data/raw/"):
    if not file.endswith(".csv"):
        continue
    
    ticker = file.replace(".csv", "")
    df = pd.read_csv(f"../data/raw/{file}", parse_dates=["Date"])
    
    # OHLC
    bad_ohlc = df[df["High"] < df["Low"]].shape[0]
    
    # duplicates
    dupes = df["Date"].duplicated().sum()
    
    # zero volume
    zero_vol = (df["Volume"] == 0).sum()
    
    # checking for outlier where daily return > 50%
    df = df.sort_values("Date")
    df["return"] = df["Close"].pct_change()
    outliers = (df["return"].abs() > 0.5).sum()
    
    print(f"{ticker}: bad_ohlc={bad_ohlc} | dupes={dupes} | zero_vol={zero_vol} | outliers={outliers}")

AAPL: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
AMD: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
AMZN: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
BAC: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
CVX: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
GOOGL: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
GS: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
JNJ: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
JPM: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
MSFT: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
NVDA: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
PFE: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
WMT: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0
XOM: bad_ohlc=0 | dupes=0 | zero_vol=0 | outliers=0


No corrupt rows, duplicate dates, zero volume days, or extreme price outliers across any ticker. The data is structurally clean out of the box.

CHECKING DATE ALIGNMENT ACROSS MULTIPLE TICKERS

In [8]:
date_sets = {}
for file in os.listdir("../data/raw/"):
    if file.endswith(".csv"):
        ticker = file.replace(".csv", "")
        df = pd.read_csv(f"../data/raw/{file}", parse_dates=["Date"])
        date_sets[ticker] = set(df["Date"])

reference = date_sets["AAPL"]
for ticker, dates in date_sets.items():
    diff = reference.symmetric_difference(dates)
    print(f"{ticker}: {len(diff)} date mismatches vs AAPL")

AAPL: 0 date mismatches vs AAPL
AMD: 0 date mismatches vs AAPL
AMZN: 0 date mismatches vs AAPL
BAC: 0 date mismatches vs AAPL
CVX: 0 date mismatches vs AAPL
GOOGL: 0 date mismatches vs AAPL
GS: 0 date mismatches vs AAPL
JNJ: 0 date mismatches vs AAPL
JPM: 0 date mismatches vs AAPL
MSFT: 0 date mismatches vs AAPL
NVDA: 0 date mismatches vs AAPL
PFE: 0 date mismatches vs AAPL
WMT: 0 date mismatches vs AAPL
XOM: 0 date mismatches vs AAPL


All 14 tickers share identical trading dates. There are no mismatches against AAPL as reference. This confirms we can safely align and compare tickers without forward-filling or interpolation.

CHECKING IF THERE ARE UNEXPECTED GAPS BETWEEN TWO TRADING DAYS

In [9]:
df = pd.read_csv("../data/raw/AAPL.csv", parse_dates=["Date"])
df = df.sort_values("Date")

gaps = df["Date"].diff().dt.days
suspicious = gaps[gaps > 4]
print(f"Suspicious gaps (>4 days):")
print(suspicious)
print(f"\nTotal suspicious gaps: {len(suspicious)}")

Suspicious gaps (>4 days):
Series([], Name: Date, dtype: float64)

Total suspicious gaps: 0


No gaps exceeding 4 days found. All absences are accounted for by weekends and market holidays.

CHECKING IF STOCK PRICES HAVE BEEN STATIONARY OR NOT

In [10]:
for file in os.listdir("../data/raw/"):
    if not file.endswith(".csv"):
        continue
    ticker = file.replace(".csv", "")
    df = pd.read_csv(f"../data/raw/{file}", parse_dates=["Date"]).sort_values("Date")
    
    price_pval = adfuller(df["Close"])[1]
    returns = df["Close"].pct_change().dropna()
    returns_pval = adfuller(returns)[1]
    
    print(f"{ticker}: price p={price_pval:.4f} ({'stationary' if price_pval < 0.05 else 'NON-stationary'}) | returns p={returns_pval:.4f} ({'stationary' if returns_pval < 0.05 else 'NON-stationary'})")

AAPL: price p=0.8969 (NON-stationary) | returns p=0.0000 (stationary)
AMD: price p=0.3575 (NON-stationary) | returns p=0.0000 (stationary)
AMZN: price p=0.7008 (NON-stationary) | returns p=0.0000 (stationary)
BAC: price p=0.4648 (NON-stationary) | returns p=0.0000 (stationary)
CVX: price p=0.6407 (NON-stationary) | returns p=0.0000 (stationary)
GOOGL: price p=0.9223 (NON-stationary) | returns p=0.0000 (stationary)
GS: price p=0.9703 (NON-stationary) | returns p=0.0000 (stationary)
JNJ: price p=0.1485 (NON-stationary) | returns p=0.0000 (stationary)
JPM: price p=0.9836 (NON-stationary) | returns p=0.0000 (stationary)
MSFT: price p=0.8134 (NON-stationary) | returns p=0.0000 (stationary)
NVDA: price p=0.9982 (NON-stationary) | returns p=0.0000 (stationary)
PFE: price p=0.5389 (NON-stationary) | returns p=0.0000 (stationary)
WMT: price p=0.9981 (NON-stationary) | returns p=0.0000 (stationary)
XOM: price p=0.8473 (NON-stationary) | returns p=0.0000 (stationary)


Prices are non-stationary across all tickers since the mean drifts upward over time and violates ARIMA's core assumption. First-differencing to daily returns achieves stationarity in all cases. This is why we predict returns throughout this project, not raw prices.

CHECKING HOW THE RETURN DISTRIBUTION IS

In [11]:
for file in os.listdir("../data/raw/"):
    if not file.endswith(".csv"):
        continue
    ticker = file.replace(".csv", "")
    df = pd.read_csv(f"../data/raw/{file}", parse_dates=["Date"]).sort_values("Date")
    returns = df["Close"].pct_change().dropna()
    
    skew = returns.skew()
    kurt = returns.kurtosis()
    
    print(f"{ticker}: skew={skew:.3f} | excess_kurtosis={kurt:.3f} | {'FAT TAILS' if kurt > 3 else 'normal-ish'}")

AAPL: skew=-0.010 | excess_kurtosis=5.508 | FAT TAILS
AMD: skew=0.372 | excess_kurtosis=2.992 | normal-ish
AMZN: skew=0.065 | excess_kurtosis=4.339 | FAT TAILS
BAC: skew=0.446 | excess_kurtosis=10.947 | FAT TAILS
CVX: skew=-0.264 | excess_kurtosis=23.781 | FAT TAILS
GOOGL: skew=-0.032 | excess_kurtosis=4.134 | FAT TAILS
GS: skew=0.360 | excess_kurtosis=10.868 | FAT TAILS
JNJ: skew=0.252 | excess_kurtosis=8.269 | FAT TAILS
JPM: skew=0.439 | excess_kurtosis=13.921 | FAT TAILS
MSFT: skew=-0.042 | excess_kurtosis=7.453 | FAT TAILS
NVDA: skew=0.356 | excess_kurtosis=4.203 | FAT TAILS
PFE: skew=0.301 | excess_kurtosis=4.181 | FAT TAILS
WMT: skew=0.099 | excess_kurtosis=14.867 | FAT TAILS
XOM: skew=0.060 | excess_kurtosis=5.203 | FAT TAILS


13 out of 14 tickers show significant excess kurtosis (fat tails), meaning extreme daily moves happen more often than a normal distribution would predict. This violates ARIMA's normality assumption and is one reason I expect XGBoost to outperform it.

CHECKING VOLUME PATTERNS

In [12]:
for file in os.listdir("../data/raw/"):
    if not file.endswith(".csv"):
        continue
    ticker = file.replace(".csv", "")
    df = pd.read_csv(f"../data/raw/{file}", parse_dates=["Date"]).sort_values("Date")
    
    mean_vol = df["Volume"].mean()
    std_vol = df["Volume"].std()
    
    # flag days where volume is more than 5 standard deviations from mean
    spikes = df[df["Volume"] > mean_vol + 5 * std_vol]
    print(f"{ticker}: mean_vol={mean_vol:.0f} | volume_spikes={len(spikes)}")

AAPL: mean_vol=94203992 | volume_spikes=6
AMD: mean_vol=64232549 | volume_spikes=3
AMZN: mean_vol=69997843 | volume_spikes=4
BAC: mean_vol=50600529 | volume_spikes=7
CVX: mean_vol=9324960 | volume_spikes=7
GOOGL: mean_vol=32626624 | volume_spikes=5
GS: mean_vol=2679214 | volume_spikes=8
JNJ: mean_vol=7980910 | volume_spikes=11
JPM: mean_vol=12903097 | volume_spikes=3
MSFT: mean_vol=27970493 | volume_spikes=5
NVDA: mean_vol=448573390 | volume_spikes=4
PFE: mean_vol=30375091 | volume_spikes=6
WMT: mean_vol=21699805 | volume_spikes=9
XOM: mean_vol=20598084 | volume_spikes=6


Volume spikes (>5 std from mean) appear in all tickers, varying between 3 to 11 per stock. These are real market events not data errors, so I will keep them. In feature engineering 
I'll use a rolling volume ratio instead of raw volume to handle these naturally. 

CLEANING AND STANDARDIZATION

In [15]:
sys.path.append("..")

from src.clean import clean_all
clean_all()

Cleaning AAPL...
  saved at processed/AAPL.csv
Cleaning MSFT...
  saved at processed/MSFT.csv
Cleaning GOOGL...
  saved at processed/GOOGL.csv
Cleaning JPM...
  saved at processed/JPM.csv
Cleaning GS...
  saved at processed/GS.csv
Cleaning BAC...
  saved at processed/BAC.csv
Cleaning XOM...
  saved at processed/XOM.csv
Cleaning CVX...
  saved at processed/CVX.csv
Cleaning AMZN...
  saved at processed/AMZN.csv
Cleaning WMT...
  saved at processed/WMT.csv
Cleaning NVDA...
  saved at processed/NVDA.csv
Cleaning AMD...
  saved at processed/AMD.csv
Cleaning JNJ...
  saved at processed/JNJ.csv
Cleaning PFE...
  saved at processed/PFE.csv

all tickers cleaned and saved.


In [16]:
from src.clean import load_and_clean

df_sample = load_and_clean("AAPL")
print(df_sample.shape)
print(df_sample.head())
print(df_sample.dtypes)
print(df_sample[["ticker", "sector"]].iloc[0])

(1509, 7)
                Close       High        Low       Open     Volume ticker  \
Date                                                                       
2019-01-02  37.469196  37.689856  36.593681  36.750278  148158800   AAPL   
2019-01-03  33.737007  34.574560  33.691926  34.161714  365248800   AAPL   
2019-01-04  35.177208  35.246017  34.118999  34.292203  234428400   AAPL   
2019-01-07  35.098911  35.312454  34.617259  35.281608  219111200   AAPL   
2019-01-08  35.768005  36.021883  35.238901  35.485657  164101200   AAPL   

           sector  
Date               
2019-01-02   tech  
2019-01-03   tech  
2019-01-04   tech  
2019-01-07   tech  
2019-01-08   tech  
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
ticker        str
sector        str
dtype: object
ticker    AAPL
sector    tech
Name: 2019-01-02 00:00:00, dtype: str
